<a href="https://colab.research.google.com/github/serggg2004/testRepository/blob/master/inn_load_from_datada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
import time
import requests
from datetime import datetime
from google.colab import files
from requests.structures import CaseInsensitiveDict

# ==================================================
# НАСТРОЙКИ ФАЙЛОВ И КЛЮЧЕЙ
# ==================================================
INPUT_FILE = "inn_list.txt"          # Файл со списком ИНН (на входе)
OUTPUT_FILE = "companies_report.txt"  # Итоговый файл с отчетами (на выходе)

# Ваши действующие ключи из ReqBin
DADATA_TOKEN = "88c5a8e4942f836f85d67f6495c25e80b26da2dd"
DADATA_SECRET = "bee181406d1246121a88743d1f5bb5d540e8532e"
# ==================================================

if not os.path.exists(INPUT_FILE):
    print(f"❌ Ошибка: Файл '{INPUT_FILE}' не найден во временном хранилище Colab!")
    print("📁 Пожалуйста, откройте вкладку с папкой (слева) и перетащите туда ваш файл с ИНН.")
else:
    # Базовый чистый URL Дадаты
    url = "https://suggestions.dadata.ru/suggestions/api/4_1/rs/findById/party"

    headers = CaseInsensitiveDict()
    headers["Content-Type"] = "application/json"
    headers["Accept"] = "application/json"
    headers["Authorization"] = f"Token {DADATA_TOKEN}"
    headers["X-Secret"] = DADATA_SECRET

    def format_date(timestamp):
        if timestamp:
            return datetime.fromtimestamp(timestamp / 1000).strftime('%d.%m.%Y')
        return "Нет данных"

    # Читаем список ИНН, убирая пустые строки
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        inn_list = [line.strip() for line in f if line.strip()]

    print(f"📋 Успешно загружен список. Найдено ИНН: {len(inn_list)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
        out_f.write(f"ОБЩИЙ ОТЧЕТ ПО ОРГАНИЗАЦИЯМ\n")
        out_f.write(f"Дата формирования: {datetime.now().strftime('%d.%m.%Y %H:%M')}\n")
        out_f.write("=" * 60 + "\n\n")

        for idx, inn in enumerate(inn_list, 1):
            print(f"[{idx}/{len(inn_list)}] Запрос к API для ИНН: {inn}...")

            # Структура тела запроса (передаем строго по ОДНОМУ ИНН в цикле)
            request_data = {
                "query": inn,
                "branch_type": "MAIN"
            }

            try:
                resp = requests.post(url, headers=headers, json=request_data)

                if resp.status_code == 200:
                    json_data = resp.json()
                    suggestions = json_data.get("suggestions", [])

                    if suggestions:
                        # ИСПРАВЛЕНО: Дадата возвращает список, берем ИНФОРМАЦИЮ О ПЕРВОЙ НАЙДЕННОЙ компании [0]
                        org = suggestions[0]
                        data_block = org.get("data", {}) or {}

                        # Извлекаем текстовые поля
                        name_full = data_block.get("name", {}).get("full_with_opf") or org.get("value") or "Нет данных"
                        ogrn = data_block.get("ogrn") or "Нет данных"
                        kpp = data_block.get("kpp") or "Нет данных"
                        okpo = data_block.get("okpo") or "Нет данных"
                        okved = data_block.get("okved") or "Нет данных"

                        # Руководство
                        management = data_block.get("management", {}) or {}
                        manager_name = management.get("name") or "Нет данных"
                        manager_post = management.get("post") or "Нет данных"
                        manager_start = format_date(management.get("start_date"))

                        # Статус и даты
                        state = data_block.get("state", {}) or {}
                        status = "Действующая" if state.get("status") == "ACTIVE" else "Ликвидирована/Неактивна"
                        reg_date = format_date(state.get("registration_date"))

                        # Адрес
                        address = org.get("unrestricted_value") or data_block.get("address", {}).get("value") or "Нет данных"

                        # Шаблон текстового блока для компании
                        report_block = f"""==================================================
ОРГАНИЗАЦИЯ №{idx} (ИНН: {inn})
==================================================
Название:          {name_full}
Статус:            {status}
Дата регистрации:  {reg_date}

Реквизиты:
--------------------------------------------------
ИНН:               {inn}
ОГРН:              {ogrn}
КПП:               {kpp}
ОКПО:              {okpo}
Основной ОКВЭД:    {okved}

Руководство:
--------------------------------------------------
Должность:         {manager_post}
ФИО:               {manager_name}
Дата вступления:   {manager_start}

Контакты и адрес:
--------------------------------------------------
Юридический адрес: {address}
==================================================\n\n"""
                        out_f.write(report_block)
                        print(f"   -> Успешно добавлена: {name_full}")
                    else:
                        out_f.write(f"⚠️ ИНН {inn}: Организация не найдена в базе Дадаты.\n\n")
                        print(f"   -> ⚠️ ИНН не найден в базе.")
                else:
                    error_msg = f"❌ ИНН {inn}: Ошибка API (Статус {resp.status_code}). Ответ сервера: {resp.text[:100]}"
                    print(f"   -> {error_msg}")
                    out_f.write(error_msg + "\n\n")

            except Exception as e:
                error_msg = f"❌ ИНН {inn}: Ошибка разбора: {str(e)}."
                print(f"   -> {error_msg}")
                out_f.write(error_msg + "\n\n")

            # Безопасная пауза 0.4 сек, чтобы не превысить лимиты тарифа DaData
            time.sleep(0.4)

    print(f"\n✅ Все ИНН обработаны! Итоговый файл '{OUTPUT_FILE}' готов.")
    print("📥 Запуск автоматического скачивания файла...")
    files.download(OUTPUT_FILE)


📋 Успешно загружен список. Найдено ИНН: 4
[1/4] Запрос к API для ИНН: 5042054367...
   -> Успешно добавлена: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ВАНДЙОРД"
[2/4] Запрос к API для ИНН: 7718718961...
   -> Успешно добавлена: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ТЕХНОИМПОРТ"
[3/4] Запрос к API для ИНН: 7718893843...
   -> Успешно добавлена: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ДАКАР"
[4/4] Запрос к API для ИНН: 7719874868...
   -> Успешно добавлена: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "ТЮНИНГ ПОЙНТ"

✅ Все ИНН обработаны! Итоговый файл 'companies_report.txt' готов.
📥 Запуск автоматического скачивания файла...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>